In [115]:
import pandas as pd
import numpy as np
from pathlib import Path

In [116]:
PROJECT_ROOT = Path.cwd().parent

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

DATA_SYMBOLIC = PROJECT_ROOT / "data" / "symbolic"
DATA_SYMBOLIC.mkdir(parents=True, exist_ok=True)

print("Processed data folder:", DATA_PROCESSED)

Processed data folder: c:\Users\ccana\Documents\Doutorado\VISEMTracking\data\processed


In [124]:
df = pd.read_csv(DATA_PROCESSED / "02_processed.csv")
df

,user_id,timestamp,trajectory_id,ftid,x,y,w,h,x_translate,y_translate,x_rotated,y_rotated,speed,angle_deg
0,14,0,ckyw6zzlj001r3867thf0fuy7,0,0.208594,0.825000,0.035937,0.037500,0.000000,0.000000,0.000000,0.000000,NaN,NaN
1,14,10,ckyw6zzlj001r3867thf0fuy7,0,0.166406,0.655833,0.035937,0.037500,-0.042187,-0.169167,0.165227,-0.055652,0.174348,-18.614460
2,14,20,ckyw6zzlj001r3867thf0fuy7,0,0.196094,0.620602,0.035937,0.037500,-0.012500,-0.204398,0.202732,-0.028893,0.046072,35.507444
3,14,30,ckyw6zzlj001r3867thf0fuy7,0,0.230022,0.603869,0.035937,0.037500,0.021429,-0.221131,0.222138,0.003581,0.037830,59.137100
4,14,40,ckyw6zzlj001r3867thf0fuy7,0,0.269531,0.617708,0.035937,0.037500,0.060937,-0.207292,0.211520,0.044074,0.041863,104.693037
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58717,54,1420,cl696lj76000r3b6gbse56a1a,0,0.415188,0.141667,0.028125,0.041667,0.005767,0.000000,0.005767,0.000000,0.000230,180.000000
58718,54,1430,cl696lj76000r3b6gbse56a1a,0,0.414959,0.141667,0.028125,0.041667,0.005537,0.000000,0.005537,0.000000,0.000230,180.000000
58719,54,1440,cl696lj76000r3b6gbse56a1a,0,0.414729,0.141667,0.028125,0.041667,0.005307,0.000000,0.005307,0.000000,0.000230,180.000000
58720,54,1450,cl696lj76000r3b6gbse56a1a,0,0.414499,0.141667,0.028125,0.041667,0.005078,0.000000,0.005078,0.000000,0.000230,180.000000


Discretização da Velocidade

In [119]:
'''# Mediana da velocidade (ignora NaN)
median_speed = float(np.nanmedian(df["speed"]))

# Limiar para movimento
eps = 1e-3 * median_speed if median_speed > 0 else 1e-6

# Considera apenas instantes em movimento
moving = df["speed"] > eps

# Quartis da velocidade (apenas quando há movimento)
if moving.any():
    q25, q75 = np.quantile(df.loc[moving, "speed"], [0.25, 0.75])
else:
    q25, q75 = 0.0, 0.0


def label_speed(v):
    if np.isnan(v):
        return np.nan
    if v <= eps:
        return "Parado"
    if v <= q25:
        return "Lento"
    if v <= q75:
        return "Medio"
    return "Rapido"

# Aplica rotulagens (vetorizado, sem apply por linha)
df["speed_label"] = df["speed"].map(label_speed)'''

'# Mediana da velocidade (ignora NaN)\nmedian_speed = float(np.nanmedian(df["speed"]))\n\n# Limiar para movimento\neps = 1e-3 * median_speed if median_speed > 0 else 1e-6\n\n# Considera apenas instantes em movimento\nmoving = df["speed"] > eps\n\n# Quartis da velocidade (apenas quando há movimento)\nif moving.any():\n    q25, q75 = np.quantile(df.loc[moving, "speed"], [0.25, 0.75])\nelse:\n    q25, q75 = 0.0, 0.0\n\n\ndef label_speed(v):\n    if np.isnan(v):\n        return np.nan\n    if v <= eps:\n        return "Parado"\n    if v <= q25:\n        return "Lento"\n    if v <= q75:\n        return "Medio"\n    return "Rapido"\n\n# Aplica rotulagens (vetorizado, sem apply por linha)\ndf["speed_label"] = df["speed"].map(label_speed)'

In [120]:
# ======================================================
# Discretização da velocidade em 5 níveis (Likert)
# SEM classe "Parado"
# ======================================================

# Remove NaN apenas para cálculo dos limiares
valid_speed = df["speed"].dropna()

# Quintis da velocidade
if not valid_speed.empty:
    q20, q40, q60, q80 = np.quantile(
        valid_speed,
        [0.20, 0.40, 0.60, 0.80]
    )
else:
    q20 = q40 = q60 = q80 = 0.0


# -----------------------------
# Rotulagem da velocidade
# -----------------------------
def label_speed_likert(v):
    if np.isnan(v):
        return np.nan
    if v <= q20:
        return "Muito_Lento"
    if v <= q40:
        return "Lento"
    if v <= q60:
        return "Medio"
    if v <= q80:
        return "Rapido"
    return "Muito_Rapido"


# Aplica rotulagem (vetorizado)
df["speed_label"] = df["speed"].map(label_speed_likert)
#df["speed_label"] = ""


Direção

In [121]:
def label_dir(a):
    if np.isnan(a):
        return np.nan

    # Normaliza ângulo para [-180, 180]
    a = ((a + 180) % 360) - 180

    if -45 < a <= 45:
        return "Leste"
    if 45 < a <= 135:
        return "Norte"
    if a > 135 or a <= -135:
        return "Oeste"
    return "Sul"

df["dir4_label"] = df["angle_deg"].map(label_dir)
#df["dir4_label"] = ""

Combinação simbólica

In [122]:
def combine_labels(speed_lbl, dir_lbl):
    if pd.isna(speed_lbl):
        return ""
    if speed_lbl == "Parado":
        return "Parado"
    if pd.isna(dir_lbl):
        return ""
    return speed_lbl + "_" + dir_lbl


df["symbol"] = [
    combine_labels(s, d)
    for s, d in zip(df["speed_label"], df["dir4_label"])
]

df.to_csv(DATA_SYMBOLIC / "01_symbolic.csv", index=False)
df

,user_id,timestamp,trajectory_id,ftid,x,y,w,h,x_translate,y_translate,x_rotated,y_rotated,speed,angle_deg,speed_label,dir4_label,symbol
0,14,0,ckyw6zzlj001r3867thf0fuy7,0,0.208594,0.825000,0.035937,0.037500,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,
1,14,10,ckyw6zzlj001r3867thf0fuy7,0,0.166406,0.655833,0.035937,0.037500,-0.042187,-0.169167,0.165227,-0.055652,0.174348,-18.614460,Muito_Rapido,Leste,Muito_Rapido_Leste
2,14,20,ckyw6zzlj001r3867thf0fuy7,0,0.196094,0.620602,0.035937,0.037500,-0.012500,-0.204398,0.202732,-0.028893,0.046072,35.507444,Muito_Rapido,Leste,Muito_Rapido_Leste
3,14,30,ckyw6zzlj001r3867thf0fuy7,0,0.230022,0.603869,0.035937,0.037500,0.021429,-0.221131,0.222138,0.003581,0.037830,59.137100,Muito_Rapido,Norte,Muito_Rapido_Norte
4,14,40,ckyw6zzlj001r3867thf0fuy7,0,0.269531,0.617708,0.035937,0.037500,0.060937,-0.207292,0.211520,0.044074,0.041863,104.693037,Muito_Rapido,Norte,Muito_Rapido_Norte
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58717,54,1420,cl696lj76000r3b6gbse56a1a,0,0.415188,0.141667,0.028125,0.041667,0.005767,0.000000,0.005767,0.000000,0.000230,180.000000,Lento,Oeste,Lento_Oeste
58718,54,1430,cl696lj76000r3b6gbse56a1a,0,0.414959,0.141667,0.028125,0.041667,0.005537,0.000000,0.005537,0.000000,0.000230,180.000000,Lento,Oeste,Lento_Oeste
58719,54,1440,cl696lj76000r3b6gbse56a1a,0,0.414729,0.141667,0.028125,0.041667,0.005307,0.000000,0.005307,0.000000,0.000230,180.000000,Lento,Oeste,Lento_Oeste
58720,54,1450,cl696lj76000r3b6gbse56a1a,0,0.414499,0.141667,0.028125,0.041667,0.005078,0.000000,0.005078,0.000000,0.000230,180.000000,Lento,Oeste,Lento_Oeste


Sequência simbólica por trajetória

In [123]:
symbolic = (
    df
    .sort_values("timestamp")
    .groupby("trajectory_id")["symbol"]
    .agg(list)
    .reset_index(name="symbolic_movement")
)

# Salva resultado
symbolic.to_csv(DATA_SYMBOLIC / "02_symbolic_per_trajectory.csv", index=False)
